# Project Overview: Semantic Email Tagging System

This project aims to develop a Semantic Email Tagging System. The system involves several key components:

1. **Email Fetching**: 
   - Using Python's `imaplib` and `email` modules to connect to an email server and fetch emails. 
   - The fetched emails include details such as the subject, sender, and body content.

2. **Email Embedding Generation**:
   - Utilizing BERT (Bidirectional Encoder Representations from Transformers) to generate embeddings for the email content. 
   - This involves tokenizing the text and extracting the mean of the last hidden state to create a numerical representation of the email content.

3. **Semantic Tagging**:
   - The ultimate goal is to use these embeddings to semantically tag emails, potentially categorizing or labeling them based on their content.

The project combines natural language processing techniques with email handling to enhance email management through semantic understanding.

In [1]:
import numpy as np
import pandas as pd
import nltk
import spacy
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.spatial.distance import cityblock, jaccard


### Fetch Email

To fetch emails, using the `imaplib` and `email` modules in Python. 



In [ ]:
import imaplib
import email
from email.header import decode_header
import configparser

def connect_to_email():

    config = configparser.ConfigParser()
    config.read("config.ini")
    
    IMAP_SERVER = config["EMAIL"]["IMAP_SERVER"]
    EMAIL_ADDRESS = config["EMAIL"]["EMAIL_ADDRESS"]
    PASSWORD = config["EMAIL"]["PASSWORD"]
    
    server = imaplib.IMAP4_SSL(IMAP_SERVER)
    server.login(EMAIL_ADDRESS, PASSWORD)
    server.select("inbox")
    return server

def fetch_emails(server, num_emails=10):
    # Search for all emails in the inbox
    status, messages = server.search(None, 'ALL')
    email_ids = messages[0].split()[-num_emails:]  # Get the last num_emails

    email_data = []
    
    for e_id in email_ids:
        # Fetch the email by ID
        status, msg_data = server.fetch(e_id, "(RFC822)")
        for response_part in msg_data:
            if isinstance(response_part, tuple):
                msg = email.message_from_bytes(response_part[1])
                
                # Decode the email subject
                subject, encoding = decode_header(msg["Subject"])[0]
                if isinstance(subject, bytes):
                    subject = subject.decode(encoding if encoding else "utf-8")
                
                # Decode the email sender
                sender, encoding = decode_header(msg.get("From"))[0]
                if isinstance(sender, bytes):
                    sender = sender.decode(encoding if encoding else "utf-8")
                
                # Get the email content
                if msg.is_multipart():
                    for part in msg.walk():
                        content_type = part.get_content_type()
                        content_disposition = str(part.get("Content-Disposition"))
                        
                        # Get the email body
                        if content_type == "text/plain" and "attachment" not in content_disposition:
                            body = part.get_payload(decode=True).decode()
                            break
                else:
                    body = msg.get_payload(decode=True).decode()

                # Store the fetched data
                email_data.append({
                    "subject": subject,
                    "sender": sender,
                    "body": body  
                })
    
    return email_data

# Example usage
server = connect_to_email()
emails = fetch_emails(server)
for email in emails:
    print(f"Subject: {email['subject']}")
    print(f"From: {email['sender']}")
    print(f"body: {email['body']}")
    print("-" * 40)


### Email Embedding Generation with BERT
Using BERT to generate embeddings for email content by tokenizing the text and extracting the mean of the last hidden state.


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

def generate_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze()

email_embeddings = [(subject, sender, generate_embedding(body)) for subject, sender, body in preprocessed_emails]

for email_item in email_embeddings:
    print("Embedding for email:", email_item[0], email_item[1])
    print("Embedding size:", email_item[2].size())
    print("-" * 50)

### Sample tags


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

tag_descriptions = {
    "Meeting (Urgent)": "This email is about an important meeting that requires immediate attention.",
    "Personal": "This email is personal and contains messages between friends or family.",
    "Notification": "This email provides a notification or update, usually from a service or company.",
    "Spam": "This email contains unsolicited and irrelevant information, winning and get suprise"
}

tag_embeddings = {tag: generate_embedding(description) for tag, description in tag_descriptions.items()}

def assign_tags(email_embedding, tag_embeddings):
    similarities = {tag: cosine_similarity(email_embedding.unsqueeze(0), tag_embedding.unsqueeze(0)).item()
                    for tag, tag_embedding in tag_embeddings.items()}
    sorted_tags = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return sorted_tags[:2]

tagged_emails = [(subject, sender, assign_tags(embedding, tag_embeddings)) for subject, sender, embedding in email_embeddings]

for email in tagged_emails:
    print(f"Email: {email[0]} from {email[1]}")
    print(f"Assigned Tags: {email[2]}")
    print("-" * 50)

In [ ]:
import imaplib
import email
from email.header import decode_header
from text_preprocessor import preprocess_text

def connect_to_email(username, password, imap_server):
    server = imaplib.IMAP4_SSL(imap_server)
    server.login(username, password)
    server.select("inbox")
    return server

def fetch_emails(server):
    status, messages = server.search(None, 'ALL')
    email_ids = messages[0].split()[-20:]  # Fetch the last 20 emails
    emails = []

    for email_id in email_ids:
        res, msg = server.fetch(email_id, "(RFC822)")
        for response_part in msg:
            if isinstance(response_part, tuple):
                msg = email.message_from_bytes(response_part[1])

                subject, encoding = decode_header(msg["Subject"])[0]
                subject = subject.decode(encoding if encoding else "utf-8") if isinstance(subject, bytes) else subject
                sender = msg.get("From")

                body = None
                if msg.is_multipart():
                    for part in msg.walk():
                        if part.get_content_type() == "text/plain":
                            body = part.get_payload(decode=True).decode()
                else:
                    body = msg.get_payload(decode=True).decode()

                if body:
                    preprocessed_body = preprocess_text(body)
                    emails.append({
                        "subject": subject,
                        "sender": sender,
                        "snippet": body[:100],
                        "preprocessed_body": preprocessed_body
                    })

    return emails
